# Answer-Match Bias Diagnostics
Analyze first answer-bearing rank, answer alias characteristics, and backend disagreement.

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().resolve()
if repo_root.name == 'retrieval_answer_eval': repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root))
from notebooks.retrieval_answer_eval.shared_setup import *

In [ ]:
fig, axes = plt.subplots(1, len(ALL_STRATEGIES), figsize=(5 * len(ALL_STRATEGIES), 4.5), sharey=True); axes = np.atleast_1d(axes)
for ax, key in zip(axes, ALL_STRATEGIES):
    frame = results_by_strategy[key].copy(); frame['rank_label'] = frame['rank'].fillna(TOP_K + 1).clip(upper=TOP_K + 1).astype(int)
    distribution = pd.crosstab(frame['decile'] + 1, frame['rank_label'], normalize='index').reindex(columns=range(1, TOP_K + 2), fill_value=0) * 100
    sns.heatmap(distribution.T, cmap='Blues', vmin=0, vmax=50, ax=ax, cbar=ax is axes[-1])
    ax.set(title=strategy_label(key), xlabel='Popularity Decile', ylabel='First Answer Rank')
    ax.set_yticklabels([str(rank) for rank in range(1, TOP_K + 1)] + [f'>{TOP_K}/Not found'], rotation=0)
fig.suptitle('First Answer-Bearing Chunk Rank by Popularity', fontweight='bold'); fig.tight_layout()
fig.savefig(IMAGES_DIR / 'answer_rank_distribution_by_decile.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
baseline = results_by_strategy['bm25_plus'][['question_id', 'decile', f'recall@{TOP_K}']].rename(columns={f'recall@{TOP_K}': 'bm25'})
comparison_key = 'ivfpq_high' if 'ivfpq_high' in results_by_strategy else ALL_STRATEGIES[-1]
paired = baseline.merge(results_by_strategy[comparison_key][['question_id', f'recall@{TOP_K}']], on='question_id', validate='one_to_one').rename(columns={f'recall@{TOP_K}': 'comparison'})
paired['outcome'] = np.select([(paired['bm25'] == 1) & (paired['comparison'] == 1), (paired['bm25'] == 1) & (paired['comparison'] == 0), (paired['bm25'] == 0) & (paired['comparison'] == 1)], ['Both', 'BM25+ only', f'{strategy_label(comparison_key)} only'], default='Neither')
disagreement = pd.crosstab(paired['decile'] + 1, paired['outcome'], normalize='index') * 100
display(disagreement.round(2))
ax = disagreement.plot(kind='bar', stacked=True, figsize=(10, 5)); ax.set(title=f'Answer Recall@{TOP_K} Agreement by Popularity', xlabel='Popularity Decile', ylabel='Questions (%)')
ax.legend(title='Outcome', bbox_to_anchor=(1.02, 1), loc='upper left'); plt.tight_layout()
plt.savefig(IMAGES_DIR / f'answer_recall_at_{TOP_K}_backend_agreement.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
alias_rows = []
for key, frame in results_by_strategy.items():
    aliases = frame['answer_texts'].apply(lambda value: list(value) if isinstance(value, (list, tuple, np.ndarray)) else [str(value)])
    alias_rows.append(pd.DataFrame({'backend': strategy_label(key), 'shortest_alias_length': aliases.apply(lambda values: min(len(str(value)) for value in values)), 'success': frame[f'recall@{TOP_K}'].values}))
alias_analysis = pd.concat(alias_rows, ignore_index=True)
alias_analysis['alias_length_band'] = pd.cut(alias_analysis['shortest_alias_length'], [0, 2, 5, 10, 20, np.inf], labels=['1-2', '3-5', '6-10', '11-20', '21+'])
summary = alias_analysis.groupby(['backend', 'alias_length_band'], observed=True)['success'].agg(['mean', 'size']).reset_index(); display(summary)
sns.catplot(data=summary, x='alias_length_band', y='mean', hue='backend', kind='point', height=4.5, aspect=1.6)
plt.ylabel(f'Answer Recall@{TOP_K}'); plt.xlabel('Shortest Gold Answer Alias Length')
plt.savefig(IMAGES_DIR / 'answer_recall_by_alias_length.png', dpi=300, bbox_inches='tight'); plt.show()